Loading the path

In [16]:
from pathlib import Path
import pandas as pd
import os 
import json

In [13]:
Fomc_sents.columns

Index(['id', 'raw_text', 'doc_type', 'clean_text', 'sentence', 'label',
       'Instruction_prompt'],
      dtype='object')

In [24]:
FOMC_announcment = Path("/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Github Repos/Project/BdF-Project/data/FED/Annotated/fomc_statements_train_1000.xls")
Fomc_sents = pd.read_excel(FOMC_announcment)
# Adding the prompt for then the subsequent JSONL transfer
Fomc_sents["Instruction_prompt"] = "Classify the following sentence from a Federal Reserve FOMC statement as Hawkish, Dovish, or Neutral. Answer with exactly one word."

Fomc_sents["label"] = Fomc_sents["label"].replace({
    "MD":"D",
    "MH":"H",
})

# Function detailing how I want my JSONL file to be outlined inorder for this to work. 
def format_example(instruction: str, input_text: str, response: str) -> str:
    return (
        f"### Instruction:\n{instruction}\n\n"
        f"### Input:\n{input_text}\n\n"
        f"### Response:\n{response}"
    )

records = []
for row in Fomc_sents[["Instruction_prompt", "sentence", "label"]].itertuples(index=False, name=None):
    instruction, sentence, label = row
    records.append({
        "instruction": instruction,
        "input": sentence,
        "response": label,
        "text": format_example(instruction, sentence, label),
    })
output_path = FOMC_announcment.parent / "fomc_statements_train_1000_3_vals.jsonl"
with open(output_path, "w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


In [25]:
"""
FOMC pipeline: Excel -> clean -> TRL-ready JSONL (train/val)
------------------------------------------------------------
Combines:
  - your Excel loading + MD/MH label collapsing
  - mojibake cleaning, dedup, conflict detection
  - conversational prompt-completion output format (TRL-native)
  - stratified train/val split

Install (if needed):  pip install pandas ftfy xlrd
"""

import json
import re
import random
from pathlib import Path
from collections import Counter

import pandas as pd

try:
    from ftfy import fix_text
    HAS_FTFY = True
except ImportError:
    HAS_FTFY = False
    print("WARNING: ftfy not installed (pip install ftfy) — using fallback fix table.")

# =======================================================================
# CONFIG
# =======================================================================
FOMC_announcement = Path(
    "/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Github Repos/Project/"
    "BdF-Project/data/FED/Annotated/fomc_statements_train_1000.xls"
)
OUTPUT_DIR = FOMC_announcement.parent
VAL_FRACTION = 0.15
SEED = 42
MIN_SENTENCE_CHARS = 15

INSTRUCTION = (
    "Classify the following sentence from a Federal Reserve FOMC statement "
    "as Hawkish, Dovish, or Neutral. Answer with exactly one word."
)

# Collapse mildly-dovish/hawkish into main classes, then map to full words
# so the model's answer matches the words offered in the instruction.
LABEL_MAP = {
    "MD": "Dovish", "MH": "Hawkish",
    "D": "Dovish", "H": "Hawkish", "N": "Neutral",
    "DOVISH": "Dovish", "HAWKISH": "Hawkish", "NEUTRAL": "Neutral",
}

MOJIBAKE_TABLE = {
    "√¢¬Ä¬ô": "'", "√¢¬Ä¬ò": "'", "√¢¬Ä¬ú": '"', "√¢¬Ä¬ù": '"',
    "√¢¬Ä¬ì": "–", "√¢¬Ä¬î": "—", "√¢¬Ä¬ë": "-", "√¢¬Ä¬¶": "...",
    "‚Äô": "'", "‚Äò": "'", "‚Äú": '"', "‚Äù": '"',
    "‚Äì": "–", "‚Äî": "—", "‚Ä¶": "...",
    "â€™": "'", "â€œ": '"', "â€\x9d": '"', "â€“": "–", "â€”": "—",
}


def clean_text(text: str) -> str:
    if HAS_FTFY:
        text = fix_text(text)
    else:
        for bad, good in MOJIBAKE_TABLE.items():
            text = text.replace(bad, good)
    return re.sub(r"\s+", " ", text).strip()


# =======================================================================
# 1. LOAD EXCEL
# =======================================================================
Fomc_sents = pd.read_excel(FOMC_announcement)
print(f"Loaded {len(Fomc_sents)} rows from {FOMC_announcement.name}")
print(f"Columns: {list(Fomc_sents.columns)}")

required = {"sentence", "label"}
missing = required - set(Fomc_sents.columns)
if missing:
    raise ValueError(f"Missing expected columns: {missing}")

# =======================================================================
# 2. CLEAN SENTENCES + NORMALIZE LABELS
# =======================================================================
Fomc_sents["sentence"] = Fomc_sents["sentence"].astype(str).map(clean_text)
Fomc_sents["label"] = (
    Fomc_sents["label"].astype(str).str.strip().str.upper().map(LABEL_MAP)
)

bad = Fomc_sents["label"].isna()
if bad.any():
    print(f"  [drop] {bad.sum()} rows with unrecognized labels")
    Fomc_sents = Fomc_sents[~bad]

short = Fomc_sents["sentence"].str.len() < MIN_SENTENCE_CHARS
if short.any():
    print(f"  [drop] {short.sum()} rows shorter than {MIN_SENTENCE_CHARS} chars")
    Fomc_sents = Fomc_sents[~short]

# =======================================================================
# 3. DEDUPLICATE
# =======================================================================
before = len(Fomc_sents)
Fomc_sents = Fomc_sents.drop_duplicates(subset=["sentence", "label"], keep="first")
print(f"  [dedup] removed {before - len(Fomc_sents)} exact duplicates")

conflicts = Fomc_sents[Fomc_sents.duplicated(subset=["sentence"], keep=False)]
if len(conflicts) > 0:
    print(f"  [WARNING] {conflicts['sentence'].nunique()} sentences have "
          f"CONFLICTING labels — dropping all {len(conflicts)} such rows:")
    for sent, group in conflicts.groupby("sentence"):
        print(f"    labels {sorted(group['label'].tolist())} :: {sent[:90]}")
    Fomc_sents = Fomc_sents[~Fomc_sents.duplicated(subset=["sentence"], keep=False)]

# =======================================================================
# 4. STRATIFIED TRAIN/VAL SPLIT
# =======================================================================
random.seed(SEED)
train_idx, val_idx = [], []
for label, group in Fomc_sents.groupby("label"):
    idx = list(group.index)
    random.shuffle(idx)
    n_val = max(1, round(len(idx) * VAL_FRACTION))
    val_idx.extend(idx[:n_val])
    train_idx.extend(idx[n_val:])

train_df = Fomc_sents.loc[train_idx].sample(frac=1, random_state=SEED)
val_df   = Fomc_sents.loc[val_idx].sample(frac=1, random_state=SEED)

# =======================================================================
# 5. WRITE — conversational prompt-completion format
# =======================================================================
def to_conversational(row) -> dict:
    return {
        "prompt": [
            {"role": "user",
             "content": f"{INSTRUCTION}\n\nSentence: {row['sentence']}"}
        ],
        "completion": [
            {"role": "assistant", "content": row["label"]}
        ],
    }

def write_jsonl(frame: pd.DataFrame, path: Path):
    with open(path, "w", encoding="utf-8") as f:
        for _, row in frame.iterrows():
            f.write(json.dumps(to_conversational(row), ensure_ascii=False) + "\n")
    print(f"  wrote {len(frame):>5} examples -> {path}")

print("\nWriting outputs:")
write_jsonl(train_df, OUTPUT_DIR / "fomc_train.jsonl")
write_jsonl(val_df,   OUTPUT_DIR / "fomc_val.jsonl")

# =======================================================================
# SUMMARY
# =======================================================================
print("\n" + "=" * 60)
print(f"Final: {len(Fomc_sents)} rows (train {len(train_df)} / val {len(val_df)})")
print("\nLabel distribution (train):")
for label, count in Counter(train_df["label"]).most_common():
    print(f"  {label:<8} {count:>5}  ({count / len(train_df):.1%})")
print("\nLabel distribution (val):")
for label, count in Counter(val_df["label"]).most_common():
    print(f"  {label:<8} {count:>5}  ({count / len(val_df):.1%})")

print("\nSample formatted example:")
print(json.dumps(to_conversational(train_df.iloc[0]), indent=2, ensure_ascii=False))

Loaded 1000 rows from fomc_statements_train_1000.xls
Columns: ['id', 'raw_text', 'doc_type', 'clean_text', 'sentence', 'label']
  [drop] 1 rows with unrecognized labels
  [drop] 3 rows shorter than 15 chars
  [dedup] removed 386 exact duplicates
  [WARNING] 50 sentences have CONFLICTING labels — dropping all 102 such rows:
    labels ['Dovish', 'Hawkish'] :: Although economic activity is likely to remain weak for a time, the Committee continues to
    labels ['Dovish', 'Neutral'] :: Although the Committee anticipates a gradual return to higher levels of resource utilizati
    labels ['Hawkish', 'Neutral'] :: Although the pace of economic recovery is likely to be moderate for a time, the Committee 
    labels ['Dovish', 'Neutral'] :: Consistent with its statutory mandate, the Committee seeks to foster maximum employment an
    labels ['Dovish', 'Neutral'] :: Currently, the unemployment rate is elevated, and measures of underlying inflation are som
    labels ['Hawkish', 'Neutral'] :: Ho